# Gold Layer ETL (DuckDB)

Builds the **Gold layer** for Kolumbus/Entur bus analytics from the cleaned Silver parquet files.

**Why DuckDB instead of Spark:** the previous version failed on Windows at the first parquet write
(`HADOOP_HOME and hadoop.home.dir are unset`, i.e. Spark needs `winutils.exe`). The Silver data is small
(87k rows), so DuckDB does the same work with no Java/Hadoop and writes the same parquet files + `gold.duckdb`.

### Silver inputs
| file | rows |
|---|---|
| `clean_stop_visits` | 87,279 |
| `expected_per_stop` | 13,780 |
| `per_stop_evidence` | 2,173 |
| `stop_event_status` | 87,092 |
| `stop_gps_status` | 86,241 |
| `time_lost_null` | 47,480 |
| `trip_vehicle` | 2,600 |

### Gold outputs
- **Dimensions:** `dim_line`, `dim_stop`, `dim_date`, `dim_hour_band`, `gold_buckets`
- **Cubes / fact:** `gold_visits_cube`, `gold_stop_cube`, `gold_stop_order_delay`, `fact_stop_visits`
- **Heatmap-ready (new):** `gold_heat_route_hour`, `gold_heat_route_quarter`, `gold_heat_weekday_hour`, `gold_heat_stop_hour`, `gold_heat_route_stop_hour`, `gold_stop_map`
- **Operations:** `gold_segment_runtime`, `gold_headway`
- **Evidence / fleet / coverage:** `gold_stop_evidence`, `gold_time_lost_reference`, `gold_fleet_activity`, `gold_stop_coverage`, `gold_event_status`
- **Audit:** `gold_data_quality`, `gold_meta`

Tables that depend on a missing Silver file, or on a key that does not match between files, are skipped (and logged), never guessed.

In [1]:
import os
from pathlib import Path
import datetime as dt
import duckdb

# ---------------- Directory Settings ----------------
SILVER_DIR = Path("../../tables/parquet_2026-10-24/silver-layer-parquet")          # folder that holds the .parquet files
GOLD_DIR = Path("../../tables/parquet_2026-10-24/gold-layer-parquet")       # cleaned outputs are written here

# ---------------- Analysis Configuration ----------------
LOCAL_TZ = "Europe/Oslo"
EARLY_S, LATE_S = -60, 300                        # on time: -60 <= delay_s < 300
DELAY_EDGES = [-120, -60, 0, 60, 180, 300, 600]   # delay histogram edges (s)
TL_EDGES    = [0, 5, 10, 20, 40]                  # time-lost histogram edges (s)
HOUR_BANDS  = {"night": (0, 5), "am_peak": (6, 8), "midday": (9, 14), "pm_peak": (15, 17), "evening": (18, 23)}
SEG_MAX_S = 1800
HEADWAY_MIN_S, HEADWAY_MAX_S = 120, 3600
BUNCH_RATIO, GAP_RATIO = 0.5, 1.5

BAND_OF_HOUR = {h: b for b, (lo, hi) in HOUR_BANDS.items() for h in range(lo, hi + 1)}
GOLD_DIR.mkdir(parents=True, exist_ok=True)
print("Silver:", SILVER_DIR.resolve())
print("Gold  :", GOLD_DIR.resolve())

Silver: D:\coding\cloud_computing\apis\Entur-Buspulse\tables\parquet_2026-10-24\silver-layer-parquet
Gold  : D:\coding\cloud_computing\apis\Entur-Buspulse\tables\parquet_2026-10-24\gold-layer-parquet


In [2]:
db_path = GOLD_DIR / "gold.duckdb"
db_path.unlink(missing_ok=True)
con = duckdb.connect(str(db_path))
con.execute("SET TimeZone='UTC'")
try:
    con.execute("LOAD icu")                               # time-zone support, normally bundled
except Exception:
    con.execute("INSTALL icu"); con.execute("LOAD icu")

LOG = []      # audit log: checks (pass/fail) and info lines
TABLES = {}   # gold table -> row count

def check(label, ok, detail=""):
    LOG.append(("check", label, bool(ok), str(detail)))
    print(f"{'PASS' if ok else 'FAIL':4s} | {label} | {detail}")

def info(label, detail=""):
    LOG.append(("info", label, True, str(detail)))
    print(f"INFO | {label} | {detail}")

def scalar(sql):
    return con.execute(sql).fetchone()[0]

def read(name, required=True):
    p = SILVER_DIR / f"{name}.parquet"
    if not p.exists():
        if required:
            raise FileNotFoundError(f"Missing required parquet file: {p}")
        print(f"Note: {p} not found, dependent tables skipped.")
        return None
    con.execute(f"CREATE OR REPLACE TEMP VIEW silver_{name} AS SELECT * FROM read_parquet('{p.as_posix()}')")
    return f"silver_{name}"

def schema(rel):
    return {r[0]: r[1].upper() for r in con.execute(f"DESCRIBE {rel}").fetchall()}

# ---- column normalisers: each returns a SQL expression ----
def ts(s, c):        # -> naive UTC TIMESTAMP
    if c not in s: return "CAST(NULL AS TIMESTAMP)"
    t = s[c]
    if t.startswith("TIMESTAMP WITH TIME ZONE") or t.startswith("TIMESTAMPTZ"): return f"(\"{c}\" AT TIME ZONE 'UTC')"
    if t.startswith("TIMESTAMP") or t == "DATE":  return f"CAST(\"{c}\" AS TIMESTAMP)"
    if t == "BIGINT":                              return f"make_timestamp(\"{c}\" // 1000)"   # epoch ns -> us
    return f"TRY_CAST(\"{c}\" AS TIMESTAMP)"

def boolean(s, c):
    if c not in s: return "FALSE"
    if s[c] == "BOOLEAN": return f"COALESCE(\"{c}\", FALSE)"
    return f"COALESCE(lower(trim(CAST(\"{c}\" AS VARCHAR))) IN ('true','1','1.0','t','yes'), FALSE)"

def num(s, c):
    if c not in s: return "CAST(NULL AS DOUBLE)"
    x = f"TRY_CAST(\"{c}\" AS DOUBLE)"
    return f"CASE WHEN isnan({x}) THEN NULL ELSE {x} END"

def label(s, c):
    if c not in s: return "'unknown'"
    x = f"CAST(\"{c}\" AS VARCHAR)"
    return f"CASE WHEN {x} IS NULL OR {x} IN ('None','nan','') THEN 'unknown' ELSE {x} END"

def to_local(expr):  # naive UTC timestamp -> naive local timestamp
    return f"(({expr} AT TIME ZONE 'UTC') AT TIME ZONE '{LOCAL_TZ}')"

HOUR_BAND_SQL = "CASE " + " ".join(
    f"WHEN local_hour BETWEEN {lo} AND {hi} THEN '{b}'" for b, (lo, hi) in HOUR_BANDS.items()) + " ELSE 'unknown' END"
WEEKDAY_NAME_SQL = "(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])[weekday + 1]"

def hist_cols(prefix, column, cond, edges):
    parts = []
    for i in range(len(edges) + 1):
        if i == 0:                c = f"{column} < {edges[0]}"
        elif i == len(edges):     c = f"{column} >= {edges[-1]}"
        else:                     c = f"{column} >= {edges[i-1]} AND {column} < {edges[i]}"
        parts.append(f"SUM(CASE WHEN ({cond}) AND {c} THEN 1 ELSE 0 END) AS {prefix}{i}")
    return ",\n  ".join(parts)

def bucket_rows(metric, prefix, edges):
    rows = []
    for i in range(len(edges) + 1):
        lo = None if i == 0 else edges[i-1]
        hi = None if i == len(edges) else edges[i]
        lbl = f"< {hi}" if lo is None else (f">= {lo}" if hi is None else f"{lo} to {hi}")
        rows.append((metric, f"{prefix}{i}", lo, hi, lbl))
    return rows

def emit(name, sql):
    """Materialise a gold table inside gold.duckdb AND write gold/<name>.parquet"""
    con.execute(f"CREATE OR REPLACE TABLE {name} AS {sql}")
    out = (GOLD_DIR / f"{name}.parquet").as_posix()
    con.execute(f"COPY {name} TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    TABLES[name] = scalar(f"SELECT COUNT(*) FROM {name}")
    print(f"  [EMIT] {name}: {TABLES[name]:,} rows")

def emit_rows(name, ddl, rows):
    con.execute(f"CREATE OR REPLACE TABLE {name} ({ddl})")
    if rows:
        ph = ",".join("?" * len(rows[0]))
        con.executemany(f"INSERT INTO {name} VALUES ({ph})", rows)
    out = (GOLD_DIR / f"{name}.parquet").as_posix()
    con.execute(f"COPY {name} TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    TABLES[name] = len(rows)
    print(f"  [EMIT] {name}: {TABLES[name]:,} rows")
print("Helpers ready.")

Helpers ready.


In [3]:
print("Staging clean_stop_visits...")
rel = read("clean_stop_visits")
s = schema(rel)

con.execute(f"""
CREATE OR REPLACE TEMP TABLE sv AS
WITH a AS (
  SELECT
    TRY_CAST(service_date AS DATE)            AS service_date,
    {label(s,'journey_ref')}                  AS journey_ref,
    {label(s,'line_ref')}                     AS line_ref,
    {label(s,'route_short_name')}             AS route_short_name,
    {label(s,'route_type_name')}              AS route_type_name,
    {label(s,'direction_ref')}                AS direction_ref,
    TRY_CAST(stop_order AS INTEGER)           AS stop_order,
    {label(s,'stop_ref')}                     AS stop_ref,
    {label(s,'stop_name')}                    AS stop_name,
    {num(s,'stop_lat')}                       AS stop_lat,
    {num(s,'stop_lon')}                       AS stop_lon,
    {ts(s,'aimed_arrival')}                   AS aimed_arrival,
    {ts(s,'actual_arrival')}                  AS actual_arrival,
    {ts(s,'aimed_departure')}                 AS aimed_departure,
    {ts(s,'actual_departure')}                AS actual_departure,
    {ts(s,'visit_time')}                      AS visit_time,
    {ts(s,'aimed_time')}                      AS aimed_time,
    {boolean(s,'is_actual')}                  AS is_actual,
    {boolean(s,'delay_outlier')}              AS delay_outlier,
    {boolean(s,'on_time')}                    AS on_time_silver,
    {num(s,'delay_s')}                        AS delay_s,
    {num(s,'delay_arrival_s')}                AS delay_arrival_s,
    {num(s,'delay_departure_s')}              AS delay_departure_s
  FROM {rel}
), l AS (
  SELECT *, {to_local('visit_time')} AS visit_local FROM a
), h AS (
  SELECT *,
    COALESCE(hour(visit_local), -1)           AS local_hour,
    COALESCE(isodow(visit_local) - 1, -1)     AS weekday,       -- Monday = 0
    COALESCE(minute(visit_local), 0)          AS local_minute
  FROM l
)
SELECT * EXCLUDE (local_minute, visit_local),
  {HOUR_BAND_SQL}                                                AS hour_band,
  CASE WHEN local_hour >= 0 THEN local_hour * 4 + local_minute // 15 ELSE -1 END AS local_quarter,
  CASE WHEN local_hour >= 0 THEN printf('%02d:%02d', local_hour, (local_minute // 15) * 15) ELSE 'unknown' END AS local_time_bin,
  (is_actual AND NOT delay_outlier AND delay_s IS NOT NULL)      AS valid
FROM h
""")

n_sv  = scalar("SELECT COUNT(*) FROM sv")
dates = [str(r[0]) for r in con.execute("SELECT DISTINCT service_date FROM sv WHERE service_date IS NOT NULL ORDER BY 1").fetchall()]
print(f"Staged clean_stop_visits: {n_sv:,} rows | service dates: {dates}")

# ---- sanity: does delay_s equal actual_departure - aimed_departure ? ----
r = con.execute("""
  SELECT avg((abs((epoch(actual_departure)-epoch(aimed_departure)) - delay_s) <= 2)::INT),
         avg((abs((epoch(actual_departure)-epoch(aimed_departure)) + delay_s) <= 2)::INT)
  FROM sv WHERE is_actual AND actual_departure IS NOT NULL AND aimed_departure IS NOT NULL AND delay_s IS NOT NULL""").fetchone()
agree, flipped = (r[0] or 0.0), (r[1] or 0.0)
check("delay_s matches actual_departure - aimed_departure", agree >= 0.85, f"agree {agree:.1%}, sign flipped {flipped:.1%}")

# ---- sanity: recomputed Oslo hour vs the local_hour already stored in silver ----
if "local_hour" in s:
    r = con.execute(f"""
      SELECT avg((CAST(c.local_hour AS INTEGER) = v.local_hour)::INT)
      FROM (SELECT {ts(s,'visit_time')} AS vt, local_hour FROM {rel}) c,
           LATERAL (SELECT COALESCE(hour({to_local('c.vt')}), -1) AS local_hour) v""").fetchone()
    info("recomputed local_hour vs silver local_hour", f"agree {r[0]:.1%}" if r[0] is not None else "n/a")


    

Staging clean_stop_visits...
Staged clean_stop_visits: 87,279 rows | service dates: ['2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27']
PASS | delay_s matches actual_departure - aimed_departure | agree 100.0%, sign flipped 0.5%
INFO | recomputed local_hour vs silver local_hour | agree 96.1%


In [4]:
probes = {
 "stop_ref with >1 name":        "SELECT COUNT(*) FROM (SELECT stop_ref FROM sv GROUP BY 1 HAVING COUNT(DISTINCT stop_name) > 1)",
 "stop_ref with >1 coord pair":  "SELECT COUNT(*) FROM (SELECT stop_ref FROM sv WHERE stop_lat IS NOT NULL GROUP BY 1 HAVING COUNT(DISTINCT (stop_lat, stop_lon)) > 1)",
 "line_ref with >1 route name":  "SELECT COUNT(*) FROM (SELECT line_ref FROM sv GROUP BY 1 HAVING COUNT(DISTINCT route_short_name) > 1)",
 "route name with >1 line_ref":  "SELECT COUNT(*) FROM (SELECT route_short_name FROM sv GROUP BY 1 HAVING COUNT(DISTINCT line_ref) > 1)",
 "rows with 'unknown' journey":  "SELECT COUNT(*) FROM sv WHERE journey_ref = 'unknown'",
 "rows with local_hour = -1":    "SELECT COUNT(*) FROM sv WHERE local_hour = -1",
 "weekday <> service weekday":   "SELECT COUNT(*) FROM sv WHERE weekday <> isodow(service_date) - 1",
 "service_date >1 day from visit":"SELECT COUNT(*) FROM sv WHERE abs(date_diff('day', service_date, CAST(visit_time AS DATE))) > 1",
}
for k, q in probes.items():
    print(f"{k:34s} {scalar(q):>10,}")

stop_ref with >1 name                       0
stop_ref with >1 coord pair                 0
line_ref with >1 route name                 0
route name with >1 line_ref                 0
rows with 'unknown' journey                 0
rows with local_hour = -1                   0
weekday <> service weekday                978
service_date >1 day from visit              7


In [5]:
if "local_hour" in s:
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE hour_diag AS
    SELECT
      TRY_CAST(service_date AS DATE)                      AS service_date,
      {ts(s,'visit_time')}                                AS visit_utc,
      {to_local(ts(s,'visit_time'))}                      AS visit_recomputed_local,
      CAST(local_hour AS INTEGER)                         AS silver_hour,
      COALESCE(hour({to_local(ts(s,'visit_time'))}), -1)  AS recomputed_hour
    FROM {rel}
    """)

    con.execute("""
    CREATE OR REPLACE TEMP TABLE hour_mismatch AS
    SELECT *,
      (silver_hour - recomputed_hour + 24) % 24           AS hour_offset,
      CAST(visit_recomputed_local AS DATE)                AS local_date,
      CAST(visit_recomputed_local AS DATE) <> service_date AS crosses_service_day
    FROM hour_diag
    WHERE silver_hour <> recomputed_hour
    """)

    n_bad = scalar("SELECT COUNT(*) FROM hour_mismatch")
    n_all = scalar("SELECT COUNT(*) FROM hour_diag")
    print(f"\nMismatches: {n_bad:,} / {n_all:,} ({n_bad/max(n_all,1):.2%})")

    print("\n1) Offset pattern (silver - recomputed, mod 24):")
    print(con.execute("""
        SELECT hour_offset, COUNT(*) n FROM hour_mismatch
        GROUP BY 1 ORDER BY n DESC""").fetchdf())

    print("\n2) Which recomputed hours / dates do they happen at:")
    print(con.execute("""
        SELECT service_date, local_date, recomputed_hour, silver_hour, COUNT(*) n
        FROM hour_mismatch
        GROUP BY ALL ORDER BY n DESC LIMIT 30""").fetchdf())

    print("\n3) Does it cross the service-day boundary (your 00 vs 24 theory)?")
    print(con.execute("""
        SELECT crosses_service_day, COUNT(*) n, MIN(visit_utc) first_utc, MAX(visit_utc) last_utc
        FROM hour_mismatch GROUP BY 1""").fetchdf())

    print("\n4) Sample rows:")
    print(con.execute("""
        SELECT service_date, visit_utc, visit_recomputed_local,
               silver_hour, recomputed_hour, hour_offset
        FROM hour_mismatch ORDER BY visit_utc LIMIT 20""").fetchdf())


Mismatches: 3,438 / 87,279 (3.94%)

1) Offset pattern (silver - recomputed, mod 24):
   hour_offset     n
0           23  3008
1            1   420
2           21     6
3           22     4

2) Which recomputed hours / dates do they happen at:
   service_date local_date  recomputed_hour  silver_hour    n
0    2026-10-24 2026-10-24               15           14  303
1    2026-10-24 2026-10-24               14           13  293
2    2026-10-24 2026-10-24               16           15  282
3    2026-10-24 2026-10-24               18           17  250
4    2026-10-24 2026-10-24               17           16  242
5    2026-10-24 2026-10-24               19           18  225
6    2026-10-24 2026-10-24               13           12  216
7    2026-10-24 2026-10-24               12           11  186
8    2026-10-24 2026-10-24               11           10  136
9    2026-10-24 2026-10-24               20           19  129
10   2026-10-24 2026-10-24               23           22   95
11   2026-1

In [6]:
print(con.execute("""
    SELECT
      avg((silver_hour = hour(visit_utc + INTERVAL 1 HOUR))::INT) AS matches_fixed_plus1,
      avg((silver_hour = hour(visit_utc))::INT)                    AS matches_utc,
      COUNT(*) n
    FROM hour_mismatch
""").fetchdf())

print(con.execute("""
    SELECT hour_offset, local_date, COUNT(*) n,
           avg((silver_hour = hour(visit_utc + INTERVAL 1 HOUR))::INT) AS fixed_plus1
    FROM hour_mismatch GROUP BY ALL ORDER BY n DESC
""").fetchdf())

   matches_fixed_plus1  matches_utc     n
0             0.874927     0.001163  3438
   hour_offset local_date     n  fixed_plus1
0           23 2026-10-24  2882          1.0
1            1 2026-10-24   375          0.0
2           23 2026-10-25   126          1.0
3            1 2026-10-25    45          0.0
4           21 2026-10-24     6          0.0
5           22 2026-10-24     3          0.0
6           22 2026-10-23     1          0.0


In [7]:
print("Building dimensions...")
emit("dim_line", """
    SELECT line_ref, ANY_VALUE(route_short_name) AS route_short_name, ANY_VALUE(route_type_name) AS route_type_name
    FROM sv GROUP BY line_ref ORDER BY line_ref""")

emit("dim_stop", """
    SELECT stop_ref, ANY_VALUE(stop_name) AS stop_name, AVG(stop_lat) AS stop_lat, AVG(stop_lon) AS stop_lon
    FROM sv GROUP BY stop_ref ORDER BY stop_ref""")

emit("dim_date", f"""
    SELECT service_date, isodow(service_date) - 1 AS weekday, strftime(service_date, '%A') AS weekday_name,
           isodow(service_date) >= 6 AS is_weekend
    FROM (SELECT DISTINCT service_date FROM sv WHERE service_date IS NOT NULL) ORDER BY service_date""")

emit_rows("dim_hour_band", "local_hour INTEGER, hour_band VARCHAR", sorted(BAND_OF_HOUR.items()))
emit_rows("gold_buckets", "metric VARCHAR, bucket VARCHAR, lo INTEGER, hi INTEGER, label VARCHAR",
          bucket_rows("delay_s", "d", DELAY_EDGES) + bucket_rows("time_lost_s", "tl", TL_EDGES))

Building dimensions...
  [EMIT] dim_line: 60 rows
  [EMIT] dim_stop: 2,174 rows
  [EMIT] dim_date: 4 rows
  [EMIT] dim_hour_band: 24 rows
  [EMIT] gold_buckets: 14 rows


In [8]:
print("Building core cubes and fact table...")
CORE = f"""
  COUNT(*) AS n_calls,
  SUM(is_actual::INT) AS n_actual,
  SUM((is_actual AND delay_outlier)::INT) AS n_outlier,
  SUM(valid::INT) AS n_valid,
  SUM((valid AND delay_s < {EARLY_S})::INT) AS n_early,
  SUM((valid AND delay_s >= {EARLY_S} AND delay_s < {LATE_S})::INT) AS n_on_time,
  SUM((valid AND delay_s >= {LATE_S})::INT) AS n_late,
  COALESCE(SUM(delay_s) FILTER (WHERE valid), 0) AS sum_delay_s"""
MEASURES = CORE + f""",
  COALESCE(SUM(delay_s * delay_s) FILTER (WHERE valid), 0) AS sum_delay_sq,
  {hist_cols('d', 'delay_s', 'valid', DELAY_EDGES)}"""

emit("gold_visits_cube", f"""
    SELECT service_date, local_hour, hour_band, weekday, line_ref, direction_ref, {MEASURES}
    FROM sv GROUP BY ALL""")

emit("gold_stop_cube", f"""
    SELECT service_date, hour_band, line_ref, stop_ref, {MEASURES}
    FROM sv GROUP BY ALL""")

emit("gold_stop_order_delay", f"""
    SELECT service_date, hour_band, line_ref, direction_ref, stop_order,
      COUNT(*) AS n_calls, SUM(valid::INT) AS n_valid,
      SUM((valid AND delay_s < {EARLY_S})::INT) AS n_early,
      SUM((valid AND delay_s >= {LATE_S})::INT) AS n_late,
      COALESCE(SUM(delay_s) FILTER (WHERE valid), 0) AS sum_delay_s,
      COALESCE(SUM(delay_s * delay_s) FILTER (WHERE valid), 0) AS sum_delay_sq
    FROM sv GROUP BY ALL""")

emit("fact_stop_visits", """
    SELECT service_date, local_hour, local_quarter, local_time_bin, hour_band, weekday, journey_ref, line_ref,
           route_short_name, direction_ref, stop_order, stop_ref, stop_name, stop_lat, stop_lon,
           aimed_arrival, actual_arrival, aimed_departure, actual_departure, visit_time,
           delay_s, delay_arrival_s, delay_departure_s, is_actual, delay_outlier, valid
    FROM sv""")

Building core cubes and fact table...


  [EMIT] gold_visits_cube: 817 rows
  [EMIT] gold_stop_cube: 13,804 rows
  [EMIT] gold_stop_order_delay: 7,877 rows
  [EMIT] fact_stop_visits: 87,279 rows


In [9]:
print("Building heatmap-ready tables...")
# One row per heatmap cell. Counts/sums are additive; avg/pct/percentiles are precomputed for plotting.
# n_valid is kept so a heatmap can mask cells with too few observations.
RATIOS = """
  sum_delay_s / NULLIF(n_valid, 0)        AS avg_delay_s,
  n_on_time::DOUBLE / NULLIF(n_valid, 0)  AS pct_on_time,
  n_early::DOUBLE   / NULLIF(n_valid, 0)  AS pct_early,
  n_late::DOUBLE    / NULLIF(n_valid, 0)  AS pct_late"""
PCTS = """,
  quantile_cont(delay_s, 0.5) FILTER (WHERE valid) AS median_delay_s,
  quantile_cont(delay_s, 0.9) FILTER (WHERE valid) AS p90_delay_s"""

def heat(name, dims, extra=""):
    # extra = additional select expressions (functionally dependent on dims, or aggregates)
    emit(name, f"""
      SELECT *, {RATIOS} FROM (
        SELECT {dims}{extra}, {CORE}{PCTS} FROM sv GROUP BY ALL)
      ORDER BY {dims}""")

# hour x route   (and 15-minute bins)
heat("gold_heat_route_hour",    "line_ref, route_short_name, local_hour")
heat("gold_heat_route_quarter", "line_ref, route_short_name, local_quarter, local_time_bin")
# weekday x hour  (rows = weekday, columns = hour)
heat("gold_heat_weekday_hour",  "weekday, local_hour", f", {WEEKDAY_NAME_SQL} AS weekday_name, COUNT(DISTINCT service_date) AS n_service_dates")
# stop x hour, and route -> stop x hour (station-hour view for one route)
heat("gold_heat_stop_hour",       "stop_ref, stop_name, local_hour")
heat("gold_heat_route_stop_hour", "line_ref, route_short_name, stop_ref, stop_name, local_hour")

# spatial: one row per stop with coordinates + aggregated metrics
ev_rel = read("per_stop_evidence", required=False)
ev_join, ev_cols = "", ""
if ev_rel:
    ev_join = f"""LEFT JOIN (SELECT CAST(stop_id AS VARCHAR) AS stop_ref, SUM(calls) AS evidence_calls,
                       SUM(stopped) AS evidence_stopped, SUM(passed) AS evidence_passed,
                       ANY_VALUE(passed_share) AS passed_share, ANY_VALUE(conclusive) AS conclusive
                FROM {ev_rel} GROUP BY 1) e ON agg.stop_ref = e.stop_ref"""
    ev_cols = ", e.evidence_calls, e.evidence_stopped, e.evidence_passed, e.passed_share, e.conclusive"
    m = scalar(f"SELECT COUNT(DISTINCT stop_ref) FROM sv WHERE stop_ref IN (SELECT CAST(stop_id AS VARCHAR) FROM {ev_rel})")
    info("per_stop_evidence.stop_id matches clean_stop_visits.stop_ref",
         f"{m:,} of {scalar('SELECT COUNT(DISTINCT stop_ref) FROM sv'):,} visited stops matched")
    dup = scalar(f"SELECT COUNT(*) - COUNT(DISTINCT stop_id) FROM {ev_rel}")
    info("per_stop_evidence duplicate stop_id rows", dup)
emit("gold_stop_map", f"""
    SELECT agg.*,
           agg.sum_delay_s / NULLIF(agg.n_valid, 0)        AS avg_delay_s,
           agg.n_on_time::DOUBLE / NULLIF(agg.n_valid, 0)  AS pct_on_time,
           agg.n_early::DOUBLE   / NULLIF(agg.n_valid, 0)  AS pct_early,
           agg.n_late::DOUBLE    / NULLIF(agg.n_valid, 0)  AS pct_late
           {ev_cols}
    FROM (SELECT stop_ref, ANY_VALUE(stop_name) AS stop_name, AVG(stop_lat) AS stop_lat, AVG(stop_lon) AS stop_lon,
                 COUNT(DISTINCT line_ref) AS n_lines, {CORE}{PCTS}
          FROM sv GROUP BY stop_ref) agg
    {ev_join}""")

Building heatmap-ready tables...
  [EMIT] gold_heat_route_hour: 783 rows
  [EMIT] gold_heat_route_quarter: 2,687 rows
  [EMIT] gold_heat_weekday_hour: 27 rows
  [EMIT] gold_heat_stop_hour: 30,811 rows
  [EMIT] gold_heat_route_stop_hour: 42,769 rows
INFO | per_stop_evidence.stop_id matches clean_stop_visits.stop_ref | 2,173 of 2,174 visited stops matched
INFO | per_stop_evidence duplicate stop_id rows | 0
  [EMIT] gold_stop_map: 2,174 rows


In [10]:
print("Building segment runtime and headway analytics...")
emit("gold_segment_runtime", f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(stop_ref) OVER w AS from_stop_ref, LAG(stop_order) OVER w AS prev_order,
                 LAG(aimed_departure) OVER w AS prev_aimed, LAG(actual_departure) OVER w AS prev_obs
          FROM v WINDOW w AS (PARTITION BY service_date, journey_ref ORDER BY stop_order)),
    s AS (SELECT *, epoch(actual_departure) - epoch(prev_obs)  AS act_s,
                    epoch(aimed_departure)  - epoch(prev_aimed) AS aim_s
          FROM p WHERE prev_order = stop_order - 1)
    SELECT service_date, hour_band, line_ref, from_stop_ref, stop_ref AS to_stop_ref,
           COUNT(*) AS n, SUM(act_s) AS sum_actual_s, SUM(aim_s) AS sum_aimed_s, SUM(act_s * act_s) AS sum_actual_sq
    FROM s WHERE act_s BETWEEN 0 AND {SEG_MAX_S} AND aim_s > 0
    GROUP BY ALL""")

emit("gold_headway", f"""
    WITH v AS (SELECT * FROM sv WHERE valid),
    p AS (SELECT *, LAG(aimed_departure) OVER w AS pa, LAG(actual_departure) OVER w AS po
          FROM v WINDOW w AS (PARTITION BY service_date, line_ref, direction_ref, stop_ref ORDER BY aimed_departure)),
    h AS (SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
                 epoch(aimed_departure) - epoch(pa) AS aim_h, epoch(actual_departure) - epoch(po) AS act_h
          FROM p WHERE pa IS NOT NULL)
    SELECT service_date, local_hour, hour_band, line_ref, direction_ref, stop_ref,
           COUNT(*) AS n_pairs, SUM(act_h) AS sum_actual_headway_s, SUM(aim_h) AS sum_aimed_headway_s,
           SUM((act_h < {BUNCH_RATIO} * aim_h)::INT) AS n_bunched,
           SUM((act_h > {GAP_RATIO} * aim_h)::INT)   AS n_large_gap
    FROM h WHERE aim_h BETWEEN {HEADWAY_MIN_S} AND {HEADWAY_MAX_S}
    GROUP BY ALL""")

Building segment runtime and headway analytics...
  [EMIT] gold_segment_runtime: 10,104 rows
  [EMIT] gold_headway: 36,240 rows


In [11]:
print("Building fleet, GPS evidence, coverage and event-status tables...")

# 1. Fleet activity (trip_vehicle)
tv = read("trip_vehicle", required=False)
if tv:
    emit("gold_fleet_activity", f"""
        SELECT TRY_CAST(service_date AS DATE) AS service_date, COUNT(*) AS n_trips,
               COUNT(DISTINCT vehicle_id) AS n_vehicles, SUM(n_stops) AS sum_stops, AVG(median_m) AS avg_median_m
        FROM {tv} GROUP BY 1 ORDER BY 1""")

# 2. GPS evidence per stop/class (stop_gps_status)
gps = read("stop_gps_status", required=False)
if gps:
    g = schema(gps)
    emit("gold_stop_evidence", f"""
        WITH ev AS (
          SELECT TRY_CAST(service_date AS DATE) AS service_date,
                 {label(g,'route_short_name')} AS route_short_name,
                 CAST(route_id AS VARCHAR) AS gtfs_route_id,
                 CAST(stop_id AS VARCHAR)  AS stop_ref,
                 {label(g,'gps_class')}    AS gps_class,
                 {num(g,'time_lost_s')}    AS time_lost_s,
                 {num(g,'p_excess')}       AS p_excess,
                 COALESCE(hour({to_local(ts(g,'visit_time'))}), -1) AS local_hour
          FROM {gps})
        SELECT service_date, {HOUR_BAND_SQL} AS hour_band, route_short_name, gtfs_route_id, stop_ref, gps_class,
               COUNT(*) AS n_calls, COUNT(time_lost_s) AS n_time_lost,
               COALESCE(SUM(time_lost_s), 0) AS sum_time_lost_s,
               SUM((p_excess <= 0.05)::INT) AS n_p_excess_le_05,
               {hist_cols('tl', 'time_lost_s', 'time_lost_s IS NOT NULL', TL_EDGES)}
        FROM ev GROUP BY ALL""")
    m = scalar("SELECT COUNT(DISTINCT stop_ref) FROM gold_stop_evidence WHERE stop_ref IN (SELECT stop_ref FROM dim_stop)")
    info("stop_gps_status.stop_id matches clean_stop_visits.stop_ref",
         f"{m:,} of {scalar('SELECT COUNT(DISTINCT stop_ref) FROM gold_stop_evidence'):,} GPS stops matched")

# 3. Time-lost reference (time_lost_null)
nul = read("time_lost_null", required=False)
if nul:
    x = num(schema(nul), "time_lost_s")
    bucket = " + ".join(f"CASE WHEN x >= {e} THEN 1 ELSE 0 END" for e in TL_EDGES)
    counts = dict(con.execute(f"SELECT b, COUNT(*) FROM (SELECT {bucket} AS b FROM (SELECT {x} AS x FROM {nul}) WHERE x IS NOT NULL) GROUP BY b").fetchall())
    emit_rows("gold_time_lost_reference", "bucket VARCHAR, n_stop_free_intervals BIGINT",
              [(f"tl{i}", int(counts.get(i, 0))) for i in range(len(TL_EDGES) + 1)])

# 4. Observed vs expected visits per stop/day (expected_per_stop) - only if keys line up
exp = read("expected_per_stop", required=False)
if exp:
    m = scalar(f"SELECT COUNT(DISTINCT stop_ref) FROM sv WHERE stop_ref IN (SELECT CAST(stop_id AS VARCHAR) FROM {exp})")
    info("expected_per_stop.stop_id matches clean_stop_visits.stop_ref", f"{m:,} visited stops matched")
    if m > 0:
        emit("gold_stop_coverage", f"""
            SELECT e.service_date, e.stop_ref, e.expected_visits,
                   COALESCE(o.n_calls, 0) AS n_observed_calls,
                   COALESCE(o.n_calls, 0)::DOUBLE / NULLIF(e.expected_visits, 0) AS observed_to_expected
            FROM (SELECT TRY_CAST(service_date AS DATE) AS service_date, CAST(stop_id AS VARCHAR) AS stop_ref,
                         SUM(expected_visits) AS expected_visits FROM {exp} GROUP BY 1, 2) e
            LEFT JOIN (SELECT service_date, stop_ref, COUNT(*) AS n_calls FROM sv GROUP BY 1, 2) o
                   USING (service_date, stop_ref)""")
    else:
        print("  gold_stop_coverage skipped: no stop_id overlap with clean_stop_visits.stop_ref")

# 5. stop_event_status: counts per status values exactly as they appear in silver (no assumed meanings)
ses = read("stop_event_status", required=False)
if ses:
    q = schema(ses)
    emit("gold_event_status", f"""
        SELECT TRY_CAST(service_date AS DATE) AS service_date, {label(q,'route_short_name')} AS route_short_name,
               CAST(stop_id AS VARCHAR) AS stop_ref, {label(q,'visit_status')} AS visit_status,
               {label(q,'final_status')} AS final_status, COUNT(*) AS n_events
        FROM {ses} GROUP BY ALL""")
    vals = con.execute("SELECT visit_status, final_status, SUM(n_events) FROM gold_event_status GROUP BY ALL ORDER BY 3 DESC").fetchall()
    info("stop_event_status values (visit_status, final_status, n)", vals)

Building fleet, GPS evidence, coverage and event-status tables...
  [EMIT] gold_fleet_activity: 2 rows
  [EMIT] gold_stop_evidence: 27,949 rows
INFO | stop_gps_status.stop_id matches clean_stop_visits.stop_ref | 2,173 of 2,173 GPS stops matched
  [EMIT] gold_time_lost_reference: 6 rows
INFO | expected_per_stop.stop_id matches clean_stop_visits.stop_ref | 2,174 visited stops matched
  [EMIT] gold_stop_coverage: 13,780 rows
  [EMIT] gold_event_status: 6,403 rows
INFO | stop_event_status values (visit_status, final_status, n) | [('observed', 'observed', 86241), ('unconfirmed', 'unconfirmed', 325), ('cancelled', 'cancelled', 303), ('unknown', 'trip_not_in_feed', 223)]


In [12]:
print("Running reconciliation audit...")
n_buckets = len(DELAY_EDGES) + 1
check("visits cube holds every call", scalar("SELECT SUM(n_calls) FROM gold_visits_cube") == n_sv)
check("stop cube holds every call",   scalar("SELECT SUM(n_calls) FROM gold_stop_cube") == n_sv)
check("fact table holds every call",  scalar("SELECT COUNT(*) FROM fact_stop_visits") == n_sv)
check("early + on_time + late = valid", scalar("SELECT SUM(n_early + n_on_time + n_late - n_valid) FROM gold_visits_cube") == 0)
check("histogram buckets add up to valid",
      scalar(f"SELECT SUM({' + '.join(f'd{i}' for i in range(n_buckets))} - n_valid) FROM gold_visits_cube") == 0)
check("stop cube valid = visits cube valid",
      scalar("SELECT SUM(n_valid) FROM gold_stop_cube") == scalar("SELECT SUM(n_valid) FROM gold_visits_cube"))
cube_sum = scalar("SELECT SUM(sum_delay_s) FROM gold_visits_cube") or 0.0
row_sum  = scalar("SELECT SUM(delay_s) FROM sv WHERE valid") or 0.0
check("sum of delays matches row-level table", abs(cube_sum - row_sum) <= 1e-4, f"cube {cube_sum:,.1f}s | rows {row_sum:,.1f}s")

for t in ["gold_heat_route_hour", "gold_heat_route_quarter", "gold_heat_weekday_hour",
          "gold_heat_stop_hour", "gold_heat_route_stop_hour", "gold_stop_map"]:
    check(f"{t} holds every call", scalar(f"SELECT SUM(n_calls) FROM {t}") == n_sv)
    check(f"{t} valid = visits cube valid",
          scalar(f"SELECT SUM(n_valid) FROM {t}") == scalar("SELECT SUM(n_valid) FROM gold_visits_cube"))

info("calls with unknown local_hour (bad/missing visit_time)", scalar("SELECT COUNT(*) FROM sv WHERE local_hour = -1"))
info("stops without coordinates in gold_stop_map", scalar("SELECT COUNT(*) FROM gold_stop_map WHERE stop_lat IS NULL OR stop_lon IS NULL"))

Running reconciliation audit...
PASS | visits cube holds every call | 
PASS | stop cube holds every call | 
PASS | fact table holds every call | 
PASS | early + on_time + late = valid | 
PASS | histogram buckets add up to valid | 
PASS | stop cube valid = visits cube valid | 
PASS | sum of delays matches row-level table | cube 12,678,367.9s | rows 12,678,367.9s
PASS | gold_heat_route_hour holds every call | 
PASS | gold_heat_route_hour valid = visits cube valid | 
PASS | gold_heat_route_quarter holds every call | 
PASS | gold_heat_route_quarter valid = visits cube valid | 
PASS | gold_heat_weekday_hour holds every call | 
PASS | gold_heat_weekday_hour valid = visits cube valid | 
PASS | gold_heat_stop_hour holds every call | 
PASS | gold_heat_stop_hour valid = visits cube valid | 
PASS | gold_heat_route_stop_hour holds every call | 
PASS | gold_heat_route_stop_hour valid = visits cube valid | 
PASS | gold_stop_map holds every call | 
PASS | gold_stop_map valid = visits cube valid | 
IN

In [13]:
print("Building dashboard-ready KPI and analytics tables...")
# Insert this cell AFTER the "fleet / GPS / coverage" cell and BEFORE the reconciliation audit cell.
# Everything is derived from sv / the existing gold tables, so local_hour is the correct per-row Europe/Oslo hour.

# ---------------- Dashboard settings ----------------
MIN_ROUTE_CALLS = 30                      # routes below this get grade 'n/a'
MIN_HOUR_CALLS  = 20                      # route-hour cells below this are ignored for "worst hour"
MIN_STOP_CALLS  = 10                      # stops below this are not ranked
HOTSPOT_PCTILE  = 0.90                    # stops above this percentile of pct_late are flagged
GRADE_CUTS = [(0.85, "A"), (0.75, "B"), (0.65, "C")]   # pct_on_time thresholds, else 'D'

GRADE_SQL = "CASE " + " ".join(f"WHEN pct_on_time >= {c} THEN '{g}'" for c, g in GRADE_CUTS) \
            + " WHEN pct_on_time IS NOT NULL THEN 'D' ELSE 'n/a' END"
BAND_ORDER_SQL = "CASE hour_band " + " ".join(f"WHEN '{b}' THEN {i}" for i, b in enumerate(HOUR_BANDS)) + " ELSE 99 END"

# 1. Route scorecard: one row per route, ranked, graded, with worst hour and headway reliability ----------
emit("gold_route_scorecard", f"""
    WITH agg AS (
      SELECT line_ref, route_short_name,
             SUM(n_calls) AS n_calls, SUM(n_valid) AS n_valid,
             SUM(sum_delay_s) / NULLIF(SUM(n_valid), 0)        AS avg_delay_s,
             SUM(n_on_time)::DOUBLE / NULLIF(SUM(n_valid), 0)  AS pct_on_time,
             SUM(n_late)::DOUBLE    / NULLIF(SUM(n_valid), 0)  AS pct_late,
             SUM(n_early)::DOUBLE   / NULLIF(SUM(n_valid), 0)  AS pct_early
      FROM gold_heat_route_hour GROUP BY line_ref, route_short_name),
    p90 AS (SELECT line_ref, quantile_cont(delay_s, 0.9) AS p90_delay_s
            FROM sv WHERE valid GROUP BY line_ref),
    hr AS (
      SELECT line_ref,
             arg_max(local_hour, avg_delay_s) AS worst_hour, MAX(avg_delay_s) AS worst_hour_delay_s,
             arg_min(local_hour, avg_delay_s) AS best_hour,  MIN(avg_delay_s) AS best_hour_delay_s
      FROM gold_heat_route_hour WHERE n_valid >= {MIN_HOUR_CALLS} AND local_hour >= 0 GROUP BY line_ref),
    hw AS (
      SELECT line_ref, SUM(n_pairs) AS n_headway_pairs,
             SUM(n_bunched)::DOUBLE   / NULLIF(SUM(n_pairs), 0) AS pct_bunched,
             SUM(n_large_gap)::DOUBLE / NULLIF(SUM(n_pairs), 0) AS pct_large_gap
      FROM gold_headway GROUP BY line_ref),
    g AS (
      SELECT a.*, p.p90_delay_s, h.worst_hour, h.worst_hour_delay_s, h.best_hour, h.best_hour_delay_s,
             w.n_headway_pairs, w.pct_bunched, w.pct_large_gap
      FROM agg a LEFT JOIN p90 p USING (line_ref) LEFT JOIN hr h USING (line_ref) LEFT JOIN hw w USING (line_ref))
    SELECT *,
           CASE WHEN n_valid >= {MIN_ROUTE_CALLS} THEN {GRADE_SQL} ELSE 'n/a' END AS grade,
           n_valid < {MIN_ROUTE_CALLS} AS low_sample,
           RANK() OVER (ORDER BY CASE WHEN n_valid >= {MIN_ROUTE_CALLS} THEN pct_on_time END DESC NULLS LAST) AS rank_on_time
    FROM g ORDER BY rank_on_time, line_ref""")

# 2. KPI by hour band (typed, for the peak vs off-peak chart and the dashboard filter) ----------
emit("gold_kpi_hour_band", f"""
    SELECT hour_band, {BAND_ORDER_SQL} AS band_order,
           SUM(n_calls) AS n_calls, SUM(n_valid) AS valid_calls,
           SUM(n_early)::DOUBLE   / NULLIF(SUM(n_valid), 0) AS pct_early,
           SUM(n_on_time)::DOUBLE / NULLIF(SUM(n_valid), 0) AS pct_on_time,
           SUM(n_late)::DOUBLE    / NULLIF(SUM(n_valid), 0) AS pct_late,
           SUM(sum_delay_s) / NULLIF(SUM(n_valid), 0)       AS mean_delay_s
    FROM gold_visits_cube GROUP BY hour_band ORDER BY band_order""")

# 3. Headline KPIs: ONE typed row for the dashboard header ----------
emit("gold_kpi", f"""
    WITH c AS (
      SELECT SUM(n_calls) AS n_calls, SUM(n_valid) AS valid_calls, SUM(n_outlier) AS n_outliers,
             SUM(n_early) AS n_early, SUM(n_on_time) AS n_on_time, SUM(n_late) AS n_late,
             SUM(sum_delay_s) AS sum_delay_s, COUNT(DISTINCT line_ref) AS n_routes
      FROM gold_visits_cube),
    q AS (SELECT quantile_cont(delay_s, 0.5) AS median_delay_s, quantile_cont(delay_s, 0.9) AS p90_delay_s
          FROM sv WHERE valid),
    hw AS (SELECT SUM(n_bunched)::DOUBLE / NULLIF(SUM(n_pairs), 0)   AS pct_bunched,
                  SUM(n_large_gap)::DOUBLE / NULLIF(SUM(n_pairs), 0) AS pct_large_gap FROM gold_headway),
    st AS (SELECT COUNT(DISTINCT stop_ref) AS n_stops FROM gold_stop_cube),
    tr AS (SELECT COUNT(DISTINCT (service_date, journey_ref)) AS n_trips FROM sv),
    d  AS (SELECT MIN(service_date) AS first_date, MAX(service_date) AS last_date,
                  COUNT(DISTINCT service_date) AS n_service_dates FROM sv),
    best  AS (SELECT route_short_name AS best_route  FROM gold_route_scorecard WHERE rank_on_time = 1 LIMIT 1),
    worst AS (SELECT route_short_name AS worst_route FROM gold_route_scorecard
              WHERE NOT low_sample ORDER BY pct_on_time ASC NULLS LAST LIMIT 1)
    SELECT c.valid_calls, c.n_calls,
           c.valid_calls::DOUBLE / NULLIF(c.n_calls, 0)  AS pct_valid,
           c.n_outliers::DOUBLE  / NULLIF(c.n_calls, 0)  AS pct_outlier,
           c.n_early::DOUBLE   / NULLIF(c.valid_calls, 0) AS pct_early,
           c.n_on_time::DOUBLE / NULLIF(c.valid_calls, 0) AS pct_on_time,
           c.n_late::DOUBLE    / NULLIF(c.valid_calls, 0) AS pct_late,
           c.sum_delay_s / NULLIF(c.valid_calls, 0)       AS mean_delay_s,
           q.median_delay_s, q.p90_delay_s, hw.pct_bunched, hw.pct_large_gap,
           c.n_routes, st.n_stops, tr.n_trips, d.n_service_dates, d.first_date, d.last_date,
           best.best_route, worst.worst_route
    FROM c, q, hw, st, tr, d LEFT JOIN best ON TRUE LEFT JOIN worst ON TRUE""")

# 4. Delay distribution in long format (histogram per route, with readable bucket labels) ----------
dcols = [f"d{i}" for i in range(len(DELAY_EDGES) + 1)]
emit("gold_delay_hist", f"""
    WITH u AS (
      SELECT line_ref, bucket, n FROM (
        SELECT line_ref, {", ".join(f"SUM({c}) AS {c}" for c in dcols)} FROM gold_visits_cube GROUP BY line_ref)
      UNPIVOT (n FOR bucket IN ({", ".join(dcols)})))
    SELECT u.line_ref, l.route_short_name, u.bucket, b.label AS bucket_label, b.lo, b.hi,
           CAST(substr(u.bucket, 2) AS INTEGER) AS bucket_order,
           u.n, u.n::DOUBLE / NULLIF(SUM(u.n) OVER (PARTITION BY u.line_ref), 0) AS share
    FROM u JOIN gold_buckets b ON b.metric = 'delay_s' AND b.bucket = u.bucket
           LEFT JOIN dim_line l USING (line_ref)
    ORDER BY u.line_ref, bucket_order""")

# 5. Route segments for the map line layer (with where delay builds up) ----------
emit("gold_route_segments", """
    WITH v AS (
      SELECT service_date, journey_ref, line_ref, route_short_name, direction_ref, stop_order, stop_ref, stop_name,
             stop_lat, stop_lon, delay_s, valid
      FROM sv WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL AND journey_ref <> 'unknown'),
    p AS (
      SELECT *, LAG(stop_ref) OVER w AS from_stop_ref, LAG(stop_name) OVER w AS from_stop_name,
                LAG(stop_lat) OVER w AS from_lat, LAG(stop_lon) OVER w AS from_lon,
                LAG(stop_order) OVER w AS prev_order, LAG(delay_s) OVER w AS prev_delay_s,
                LAG(valid) OVER w AS prev_valid
      FROM v WINDOW w AS (PARTITION BY service_date, journey_ref ORDER BY stop_order))
    SELECT line_ref, route_short_name, direction_ref,
           from_stop_ref, from_stop_name, stop_ref AS to_stop_ref, stop_name AS to_stop_name,
           from_lat, from_lon, stop_lat AS to_lat, stop_lon AS to_lon,
           COUNT(*) AS n_trips,
           COUNT(*) FILTER (WHERE valid) AS n_valid,
           AVG(delay_s) FILTER (WHERE valid) AS avg_delay_s,
           AVG((delay_s >= 300)::INT) FILTER (WHERE valid) AS pct_late,
           AVG(delay_s - prev_delay_s) FILTER (WHERE valid AND prev_valid) AS avg_delay_gain_s
    FROM p WHERE prev_order = stop_order - 1
    GROUP BY ALL""")

# 6. Stop ranking and hotspots ----------
thr = scalar(f"SELECT quantile_cont(pct_late, {HOTSPOT_PCTILE}) FROM gold_stop_map WHERE n_valid >= {MIN_STOP_CALLS}")
thr = float(thr) if thr is not None else 1.0
emit("gold_stop_ranking", f"""
    SELECT stop_ref, stop_name, stop_lat, stop_lon, n_lines, n_valid,
           avg_delay_s, median_delay_s, p90_delay_s, pct_on_time, pct_late,
           RANK() OVER (ORDER BY pct_late DESC)    AS rank_late,
           RANK() OVER (ORDER BY avg_delay_s DESC) AS rank_delay,
           pct_late >= {thr} AS is_hotspot
    FROM gold_stop_map WHERE n_valid >= {MIN_STOP_CALLS} ORDER BY rank_late""")

# 7. Trip summary: how much delay a trip picks up between its first and last stop ----------
emit("gold_trip_summary", f"""
    WITH t AS (
      SELECT service_date, journey_ref, ANY_VALUE(line_ref) AS line_ref, ANY_VALUE(route_short_name) AS route_short_name,
             ANY_VALUE(direction_ref) AS direction_ref, COUNT(*) AS n_stops, COUNT(*) FILTER (WHERE valid) AS n_valid_stops,
             MIN(visit_time) AS trip_start_utc, MAX(visit_time) AS trip_end_utc,
             arg_min(delay_s, stop_order) FILTER (WHERE valid) AS first_delay_s,
             arg_max(delay_s, stop_order) FILTER (WHERE valid) AS last_delay_s,
             MAX(delay_s) FILTER (WHERE valid) AS max_delay_s,
             AVG(delay_s) FILTER (WHERE valid) AS avg_delay_s,
             COALESCE(hour({to_local('MIN(visit_time)')}), -1) AS start_local_hour
      FROM sv GROUP BY service_date, journey_ref)
    SELECT *, last_delay_s - first_delay_s AS delay_gain_s,
           CASE {" ".join(f"WHEN start_local_hour BETWEEN {lo} AND {hi} THEN '{b}'" for b, (lo, hi) in HOUR_BANDS.items())} ELSE 'unknown' END AS start_hour_band
    FROM t""")

# 8. Optional: H3 zone columns for hexbin layer / "by zone" filter (needs the community h3 extension) ----------
try:
    con.execute("INSTALL h3 FROM community"); con.execute("LOAD h3")
    emit("gold_stop_h3", """
        SELECT stop_ref, stop_name, stop_lat, stop_lon,
               h3_h3_to_string(h3_latlng_to_cell(stop_lat, stop_lon, 7)) AS h3_r7,
               h3_h3_to_string(h3_latlng_to_cell(stop_lat, stop_lon, 9)) AS h3_r9
        FROM dim_stop WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL""")
except Exception as e:
    info("gold_stop_h3 skipped (h3 extension unavailable)", str(e)[:150])

# ---------------- Reconciliation checks for the new tables ----------------
vc = scalar("SELECT SUM(n_valid) FROM gold_visits_cube")
check("scorecard valid = visits cube valid",  scalar("SELECT SUM(n_valid) FROM gold_route_scorecard") == vc)
check("kpi is a single row",                  scalar("SELECT COUNT(*) FROM gold_kpi") == 1)
check("kpi valid = visits cube valid",        scalar("SELECT valid_calls FROM gold_kpi") == vc)
check("hour-band kpi valid = visits cube valid", scalar("SELECT SUM(valid_calls) FROM gold_kpi_hour_band") == vc)
check("delay histogram adds up to valid",     scalar("SELECT SUM(n) FROM gold_delay_hist") == vc)
check("trip summary holds every stop call",   scalar("SELECT SUM(n_stops) FROM gold_trip_summary") == n_sv)
info("route segments / hotspot stops", f"{scalar('SELECT COUNT(*) FROM gold_route_segments'):,} segments | "
     f"{scalar('SELECT COUNT(*) FROM gold_stop_ranking WHERE is_hotspot'):,} hotspot stops (pct_late >= {thr:.1%})")

Building dashboard-ready KPI and analytics tables...


  [EMIT] gold_route_scorecard: 60 rows
  [EMIT] gold_kpi_hour_band: 5 rows
  [EMIT] gold_kpi: 1 rows
  [EMIT] gold_delay_hist: 480 rows
  [EMIT] gold_route_segments: 3,934 rows
  [EMIT] gold_stop_ranking: 1,522 rows
  [EMIT] gold_trip_summary: 2,623 rows
  [EMIT] gold_stop_h3: 2,174 rows
PASS | scorecard valid = visits cube valid | 
PASS | kpi is a single row | 
PASS | kpi valid = visits cube valid | 
PASS | hour-band kpi valid = visits cube valid | 
PASS | delay histogram adds up to valid | 
PASS | trip summary holds every stop call | 
INFO | route segments / hotspot stops | 3,934 segments | 153 hotspot stops (pct_late >= 37.8%)


In [14]:
print("Building map and hexagon tables for the dashboard...")
# Insert AFTER gold_dashboard_cell.py and BEFORE the reconciliation audit cell.
# Needs: sv, dim_stop, emit, scalar, check, info, EARLY_S, LATE_S, SEG_MAX_S (all defined earlier in the notebook).

# 1. Road segments by hour: one row per (date, route, from-stop -> to-stop, hour of the to-stop visit) ----------
emit("gold_segments_hour", f"""
    WITH v AS (
      SELECT service_date, journey_ref, line_ref, route_short_name, stop_order, stop_ref, stop_name,
             stop_lat, stop_lon, local_hour, delay_s, valid, aimed_departure, actual_departure
      FROM sv WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL AND journey_ref <> 'unknown'),
    p AS (
      SELECT *, LAG(stop_ref) OVER w AS from_stop_ref, LAG(stop_name) OVER w AS from_stop_name,
                LAG(stop_lat) OVER w AS from_lat, LAG(stop_lon) OVER w AS from_lon,
                LAG(stop_order) OVER w AS prev_order, LAG(valid) OVER w AS prev_valid,
                LAG(aimed_departure) OVER w AS prev_aimed, LAG(actual_departure) OVER w AS prev_actual
      FROM v WINDOW w AS (PARTITION BY service_date, journey_ref ORDER BY stop_order)),
    s AS (
      SELECT *, epoch(actual_departure) - epoch(prev_actual) AS act_s,
                epoch(aimed_departure)  - epoch(prev_aimed)  AS aim_s,
                2 * 6371000 * asin(sqrt(pow(sin(radians(stop_lat - from_lat) / 2), 2)
                    + cos(radians(from_lat)) * cos(radians(stop_lat)) * pow(sin(radians(stop_lon - from_lon) / 2), 2))) AS dist_m
      FROM p WHERE prev_order = stop_order - 1),
    r AS (SELECT *, (valid AND prev_valid AND act_s BETWEEN 0 AND {SEG_MAX_S} AND aim_s > 0) AS rt_ok FROM s)
    SELECT service_date, line_ref, route_short_name,
           from_stop_ref, from_stop_name, stop_ref AS to_stop_ref, stop_name AS to_stop_name,
           from_lat, from_lon, stop_lat AS to_lat, stop_lon AS to_lon, local_hour,
           ANY_VALUE(dist_m) AS dist_m,
           COUNT(*) AS n_trips,
           COUNT(*) FILTER (WHERE valid) AS n_valid,
           COALESCE(SUM(delay_s) FILTER (WHERE valid), 0) AS sum_delay_s,
           COUNT(*) FILTER (WHERE valid AND delay_s >= {EARLY_S} AND delay_s < {LATE_S}) AS n_on_time,
           COUNT(*) FILTER (WHERE valid AND delay_s >= {LATE_S}) AS n_late,
           COUNT(*) FILTER (WHERE rt_ok) AS n_rt,
           COALESCE(SUM(act_s)  FILTER (WHERE rt_ok), 0) AS sum_runtime_s,
           COALESCE(SUM(dist_m) FILTER (WHERE rt_ok), 0) AS sum_dist_m
    FROM r GROUP BY ALL""")

# 2. Stop -> H3 cell mapping (res 8 ~ 0.46 km edge, res 9 ~ 0.17 km edge) ----------
def _map_stops_to_h3():
    try:                                                         # preferred: DuckDB community extension
        con.execute("INSTALL h3 FROM community"); con.execute("LOAD h3")
        con.execute("""
            CREATE OR REPLACE TEMP TABLE stop_h3 AS
            SELECT stop_ref,
                   h3_h3_to_string(h3_latlng_to_cell(stop_lat, stop_lon, 8)) AS h3_r8,
                   h3_h3_to_string(h3_latlng_to_cell(stop_lat, stop_lon, 9)) AS h3_r9
            FROM dim_stop WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL""")
        return "duckdb h3 extension"
    except Exception:
        pass
    try:                                                         # fallback: `pip install h3`
        import h3 as _h3
        fn = getattr(_h3, "latlng_to_cell", None) or _h3.geo_to_h3
        d = con.execute("SELECT stop_ref, stop_lat, stop_lon FROM dim_stop "
                        "WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL").fetchdf()
        d["h3_r8"] = [fn(a, b, 8) for a, b in zip(d.stop_lat, d.stop_lon)]
        d["h3_r9"] = [fn(a, b, 9) for a, b in zip(d.stop_lat, d.stop_lon)]
        con.register("stop_h3_df", d[["stop_ref", "h3_r8", "h3_r9"]])
        con.execute("CREATE OR REPLACE TEMP TABLE stop_h3 AS SELECT * FROM stop_h3_df")
        return "python h3 package"
    except Exception:
        return None

h3_method = _map_stops_to_h3()

# 3. Hexagon heat by hour and route ----------
if h3_method:
    emit("gold_hex_hour", f"""
        SELECT v.service_date, v.line_ref, v.route_short_name, v.local_hour, t.h3_r8, t.h3_r9,
               COUNT(*) AS n_calls,
               COUNT(*) FILTER (WHERE v.valid) AS n_valid,
               COALESCE(SUM(v.delay_s) FILTER (WHERE v.valid), 0) AS sum_delay_s,
               COUNT(*) FILTER (WHERE v.valid AND v.delay_s >= {EARLY_S} AND v.delay_s < {LATE_S}) AS n_on_time,
               COUNT(*) FILTER (WHERE v.valid AND v.delay_s >= {LATE_S}) AS n_late
        FROM sv v JOIN stop_h3 t USING (stop_ref)
        GROUP BY ALL""")
    check("hex table holds every located call",
          scalar("SELECT SUM(n_calls) FROM gold_hex_hour") ==
          scalar("SELECT COUNT(*) FROM sv WHERE stop_lat IS NOT NULL AND stop_lon IS NOT NULL"), h3_method)
else:
    info("gold_hex_hour skipped", "neither the DuckDB h3 extension nor the python h3 package is available")

info("segments by hour", f"{scalar('SELECT COUNT(*) FROM gold_segments_hour'):,} rows | "
     f"{scalar('SELECT COUNT(DISTINCT (from_stop_ref, to_stop_ref)) FROM gold_segments_hour'):,} unique stop pairs")

Building map and hexagon tables for the dashboard...
  [EMIT] gold_segments_hour: 43,511 rows
  [EMIT] gold_hex_hour: 28,837 rows
PASS | hex table holds every located call | duckdb h3 extension
INFO | segments by hour | 43,511 rows | 2,326 unique stop pairs


In [15]:
print("Publishing audit log, metadata and headline metrics...")
emit_rows("gold_data_quality", "kind VARCHAR, label VARCHAR, ok BOOLEAN, detail VARCHAR", LOG)

h = con.execute("SELECT SUM(n_valid), SUM(n_early), SUM(n_on_time), SUM(n_late), SUM(sum_delay_s) FROM gold_visits_cube").fetchone()
if h and h[0]:
    print("HEADLINE PERFORMANCE")
    print(f"  Valid calls : {int(h[0]):,}")
    print(f"  Early       : {h[1] / h[0]:.1%}")
    print(f"  On time     : {h[2] / h[0]:.1%}")
    print(f"  Late        : {h[3] / h[0]:.1%}")
    print(f"  Mean delay  : {h[4] / h[0]:.1f} s")

n_pass = sum(1 for k, _, ok, _ in LOG if k == "check" and ok)
n_fail = sum(1 for k, _, ok, _ in LOG if k == "check" and not ok)
meta = {"built_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "engine": f"duckdb {duckdb.__version__}",
        "service_dates": ",".join(dates), "timezone": LOCAL_TZ,
        "on_time_rule": f"{EARLY_S} <= delay_s < {LATE_S}", "valid_rule": "is_actual AND NOT delay_outlier AND delay_s IS NOT NULL",
        "checks_passed": str(n_pass), "checks_failed": str(n_fail)}
meta.update({f"rows_{k}": str(v) for k, v in TABLES.items()})
emit_rows("gold_meta", "key VARCHAR, value VARCHAR", list(meta.items()))
con.close()
print(f"\ngold.duckdb written with {len(TABLES)} tables -> {db_path.resolve()}")

Publishing audit log, metadata and headline metrics...
  [EMIT] gold_data_quality: 37 rows
HEADLINE PERFORMANCE
  Valid calls : 87,241
  Early       : 6.0%
  On time     : 78.5%
  Late        : 15.5%
  Mean delay  : 145.3 s
  [EMIT] gold_meta: 41 rows

gold.duckdb written with 34 tables -> D:\coding\cloud_computing\apis\Entur-Buspulse\tables\parquet_2026-10-24\gold-layer-parquet\gold.duckdb


In [16]:
failed = [l for k, l, ok, _ in LOG if k == "check" and not ok]
print(f"SUCCESS: all {n_pass} reconciliation checks passed. Gold layer ready at {GOLD_DIR.resolve()}" if not failed
      else f"WARNING: {len(failed)} check(s) failed: {failed}")
assert not failed, f"Failed checks: {failed}"

SUCCESS: all 27 reconciliation checks passed. Gold layer ready at D:\coding\cloud_computing\apis\Entur-Buspulse\tables\parquet_2026-10-24\gold-layer-parquet
